# 👥 Notebook 2: User-Based Collaborative Filtering

**Mục tiêu:**
- Hiểu ý tưởng: "Người dùng giống nhau sẽ thích phim giống nhau"
- Tính similarity giữa users (Cosine)
- Tìm K nearest neighbors
- Gợi phim dựa trên neighbors

**Thuật toán:** KNN with Means — dùng thư viện `scikit-surprise`

## 1. Lý thuyết: User-Based CF

```
Ý tưởng: Nếu User A và User B đều thích phim X, Y, Z
         → Họ có "gu" giống nhau
         → Phim mà User B thích (mà A chưa xem)
           rất có thể User A cũng sẽ thích

Các bước:
  1. Tính similarity giữa User A và tất cả users khác
  2. Chọn K users giống A nhất (K-Nearest Neighbors)
  3. Dự đoán rating của A cho phim i:
     r̂(A,i) = r̄_A + Σ(sim(A,B) × (r_B,i - r̄_B)) / Σ|sim(A,B)|
```

In [ ]:
# Import
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from surprise import KNNWithMeans, Dataset, Reader, accuracy
from surprise.model_selection import train_test_split, cross_validate
import warnings
warnings.filterwarnings('ignore')

# Load data
ratings = pd.read_csv('data/processed/ratings_clean.csv')
movies  = pd.read_csv('data/processed/movies_clean.csv')

print(f'Loaded: {len(ratings):,} ratings, {len(movies):,} movies')

## 2. Chuẩn bị data cho scikit-surprise

scikit-surprise yêu cầu data theo định dạng: `[userId, movieId, rating]`

In [ ]:
# Định nghĩa rating scale
reader = Reader(rating_scale=(1, 5))

# Load vào Surprise format
data = Dataset.load_from_df(
    ratings[['userId', 'movieId', 'rating']], 
    reader
)

# Chia train/test (80/20)
trainset, testset = train_test_split(data, test_size=0.2, random_state=42)

print(f'Train: {trainset.n_ratings:,} ratings')
print(f'Test:  {len(testset):,} ratings')
print(f'Users in train: {trainset.n_users:,}')
print(f'Movies in train: {trainset.n_items:,}')

## 3. Train User-Based CF Model

**Tham số:**
- `k=40`: Số neighbors tối đa
- `sim_option={'name':'cosine'}`: Dùng Cosine similarity
- `user_based=True`: Đây là User-Based (không phải Item-Based)

In [ ]:
# User-Based CF với Cosine Similarity
print('🔧 Đang train User-Based CF (k=40, cosine)...')

model = KNNWithMeans(
    k=40,
    sim_option={
        'name': 'cosine',      # Cosine similarity
        'user_based': True     # ← User-Based!
    },
    verbose=False
)

model.fit(trainset)
print('✅ Train xong!')

## 4. Đánh giá trên Test Set

In [ ]:
# Dự đoán trên test set
predictions = model.test(testset)

# Tính RMSE và MAE
rmse = accuracy.rmse(predictions, verbose=False)
mae  = accuracy.mae(predictions, verbose=False)

print(f'📊 User-Based CF Results:')
print(f'   RMSE: {rmse:.4f}')
print(f'   MAE:  {mae:.4f}')

In [ ]:
# Xem 10 predictions đầu
print('Ví dụ predictions:')
for p in predictions[:10]:
    print(f'  User {p.uid} | Movie {p.iid} | True={p.r_ui:.0f} | Pred={p.est:.2f}')

## 5. Cross-Validation

Đánh giá robust hơn bằng 5-fold CV:

In [ ]:
print('🔄 5-Fold Cross-Validation...')
cv_results = cross_validate(
    model, data, measures=['RMSE', 'MAE'], cv=5, verbose=True
)

print(f"\n✅ CV RMSE: {cv_results['test_rmse'].mean():.4f} ± {cv_results['test_rmse'].std():.4f}")
print(f"✅ CV MAE:  {cv_results['test_mae'].mean():.4f} ± {cv_results['test_mae'].std():.4f}")

## 6. Gợi ý phim cho User cụ thể

In [ ]:
def recommend_user_cf(model, user_id, ratings_df, movies_df, top_n=10):
    """Gợi phim bằng User-Based CF."""
    # Phim đã xem
    user_movies = ratings_df[ratings_df['userId'] == user_id]['movieId'].values
    # Tất cả phim trong dataset
    all_movies = ratings_df['movieId'].unique()
    # Phim chưa xem
    unseen = [m for m in all_movies if m not in user_movies]

    # Dự đoán score
    scores = {}
    for movie_id in unseen:
        pred = model.predict(user_id, movie_id)
        scores[movie_id] = pred.est

    # Sắp xếp giảm dần
    sorted_scores = sorted(scores.items(), key=lambda x: x[1], reverse=True)

    # Lấy top N
    results = []
    for mid, score in sorted_scores[:top_n]:
        title = movies_df[movies_df['movieId'] == mid]['title'].values
        if len(title) > 0:
            results.append({'movieId': mid, 'title': title[0], 'score': round(score, 2)})
    return results


# Demo: Gợi cho user 1
user_id = 1
recs = recommend_user_cf(model, user_id, ratings, movies, top_n=10)

print(f'🎬 Top 10 gợi ý cho User {user_id}:')
pd.DataFrame(recs)

In [ ]:
# Thông tin user 1
user1_ratings = ratings[ratings['userId'] == 1]
user1_movies = movies[movies['movieId'].isin(user1_ratings['movieId'])]
user1_movies = user1_movies.merge(user1_ratings[['movieId','rating']], on='movieId')
user1_movies = user1_movies.sort_values('rating', ascending=False)

print(f'User 1 đã đánh giá {len(user1_movies)} phim')
print(f'Rating trung bình: {user1_movies["rating"].mean():.2f}')
print('\nPhim được đánh giá cao nhất:')
user1_movies.head(5)[['title','rating']]

## 7. Thử nghiệm với các giá trị K khác nhau

In [ ]:
k_values = [10, 20, 40, 60, 80]
rmse_scores = []

print('Thử nghiệm K khác nhau...')
for k in k_values:
    m = KNNWithMeans(k=k, sim_option={'name': 'cosine', 'user_based': True}, verbose=False)
    cv = cross_validate(m, data, measures=['RMSE'], cv=3, verbose=False)
    rmse_scores.append(cv['test_rmse'].mean())
    print(f'  K={k:3d} → RMSE={cv["test_rmse"].mean():.4f}')

# Vẽ biểu đồ
plt.figure(figsize=(8, 4))
plt.plot(k_values, rmse_scores, marker='o', color='steelblue', linewidth=2, markersize=8)
plt.xlabel('K (số neighbors)')
plt.ylabel('RMSE')
plt.title('User-Based CF: RMSE theo K')
plt.grid(True, alpha=0.3)
plt.savefig('results/charts/02_user_cf_rmse_vs_k.png', dpi=150, bbox_inches='tight')
plt.show()

## 8. So sánh Similarity Metrics

In [ ]:
sims = ['cosine', 'pearson', 'msd']
print('So sánh similarity metrics (K=40):')

for sim_name in sims:
    m = KNNWithMeans(k=40, sim_option={'name': sim_name, 'user_based': True}, verbose=False)
    cv = cross_validate(m, data, measures=['RMSE', 'MAE'], cv=3, verbose=False)
    print(f'  {sim_name:8s} → RMSE={cv["test_rmse"].mean():.4f}, MAE={cv["test_mae"].mean():.4f}')

## 9. Tổng kết

**Kết quả User-Based CF:**
- RMSE ≈ 0.87–0.90
- MAE ≈ 0.68–0.70

**Nhận xét:**
- User-Based CF hoạt động tốt nhưng chậm vì cần tính similarity giữa tất cả users
- K quá nhỏ → thiếu thông tin, K quá lớn → nhiễu
- Thường dùng K=20–40 cho MovieLens